# UMAP Video Generator

Generate combined videos that show a dataset view alongside the saved UMAP watershed density map, with a red marker indicating the current classified position.

Use the configuration cell below to change dataset selection, paths, and prototype settings. Leave the implementation cells unchanged unless you are modifying the rendering logic.

## Configuration

Edit this section only if you want to change notebook inputs or output behavior.

What you can change here:
- `DATASET_SELECTION` to choose `hive1_upper`, `hive2_upper`, `hive2_lower`, or `all`.
- `PROTOTYPE_FRAME_LIMIT` to control how many source-video frames are processed during prototyping.
- `LAYOUT`, `FPS`, and output paths if you want a different video layout or destination.
- The `video_alias` entries if your alias-backed source videos move.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
RESULTS_DIR = PROJECT_DIR / "Results" / "WideNormalisation" / "UMAP"
PROJECTIONS_DIR = PROJECT_DIR / "Results" / "WideNormalisation" / "Projections"
WSHED_PATH = RESULTS_DIR / "zVals_wShed_groups.mat"
OUTPUT_DIR = PROJECT_DIR / "outputVideos" / "umap_video"
LAYOUT = "side_by_side"  # or "top_bottom"
FPS = 20
VIDEO_PREFIX = "dataset_umap"
FRAME_LIMIT = None
BUCKET = "ObsHiveABC"
DOWNLOAD_RESOLUTION = 600
SOURCE_VIDEO_SCALE = 0.4
UMAP_PANEL_FIGSIZE = (8, 8)
UMAP_PANEL_DPI = 180
UMAP_PANEL_TITLE_FONTSIZE = 16
UMAP_PANEL_AXIS_LABELSIZE = 14
UMAP_PANEL_TICK_LABELSIZE = 12

# Dataset selection and alias-backed source videos
DATASET_CONFIGS = {
    "hive1_upper": {
        "hive_nb": 1,
        "ihl": "upper",
        "start_ts": pd.Timestamp('2024-09-04 16:00:00').tz_localize("UTC"),
        "projection_source_id": 1,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset2",
    },
    "hive2_upper": {
        "hive_nb": 2,
        "ihl": "upper",
        "start_ts": pd.Timestamp('2024-10-02 18:00:00').tz_localize("UTC"),
        "projection_source_id": 0,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset1",
    },
    "hive2_lower": {
        "hive_nb": 2,
        "ihl": "lower",
        "start_ts": pd.Timestamp('2024-10-02 18:00:00').tz_localize("UTC"),
        "projection_source_id": 2,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset1",
    },
}

DATASET_SELECTION = "hive1_upper"  # choose one key above or "all"
selected_datasets = [DATASET_SELECTION] if DATASET_SELECTION != "all" else list(DATASET_CONFIGS.keys())

## Imports

Import libraries for data handling, plotting, image processing, and video creation. The helper module provides the reusable UMAP/video frame functions.

In [2]:
import sys
import importlib
import numpy as np
import cv2
from tqdm.auto import tqdm
import imageio.v2 as imageio

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import umap_video_utils as umap_video_utils_module
umap_video_utils_module = importlib.reload(umap_video_utils_module)

compose_panels = umap_video_utils_module.compose_panels
load_umap_points_from_projection_file = umap_video_utils_module.load_umap_points_from_projection_file
load_watershed_artifacts = umap_video_utils_module.load_watershed_artifacts
render_umap_panel = umap_video_utils_module.render_umap_panel
generateVideoFromFrames = umap_video_utils_module.generateVideoFromFrames
resolve_macos_alias = umap_video_utils_module.resolve_macos_alias
load_projection_timeline = umap_video_utils_module.load_projection_timeline
load_umap_data = umap_video_utils_module.load_umap_data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Load UMAP Embedding, Density Map, and Classifier Outputs

Load the precomputed watershed density background and the border coordinates needed to place the current timestamp on the UMAP map.

In [3]:
import importlib
import umap_video_utils as umap_video_utils_module
importlib.reload(umap_video_utils_module)
load_watershed_artifacts = umap_video_utils_module.load_watershed_artifacts
wshed_artifacts = load_watershed_artifacts(str(WSHED_PATH))
wshed_artifacts


{'density': array([[3.95457926e-07, 4.39353110e-07, 4.88093835e-07, ...,
         2.88799338e-07, 3.20549384e-07, 3.55985252e-07],
        [3.82759633e-07, 4.25195774e-07, 4.72338213e-07, ...,
         2.79765663e-07, 3.10401453e-07, 3.44619474e-07],
        [3.70710692e-07, 4.11755120e-07, 4.57374163e-07, ...,
         2.71218728e-07, 3.00788448e-07, 3.33842653e-07],
        ...,
        [4.37487087e-07, 4.86168544e-07, 5.40159151e-07, ...,
         3.18845865e-07, 3.54231819e-07, 3.93649770e-07],
        [4.22818398e-07, 4.69835842e-07, 5.22000214e-07, ...,
         3.08337202e-07, 3.42462015e-07, 3.80497512e-07],
        [4.08809662e-07, 4.54231793e-07, 5.04646328e-07, ...,
         2.98322390e-07, 3.31235322e-07, 3.67943553e-07]]),
 'xx': array([[-102.27294922, -101.93762807, -101.60230693, -101.26698578,
         -100.93166464, -100.59634349, -100.26102235,  -99.9257012 ,
          -99.59038006,  -99.25505891,  -98.91973777,  -98.58441662,
          -98.24909548,  -97.91377433,  -

## Create Frame Overlay for the Current Classification Point

Draw a red dot on the UMAP density map at the classified position for the current frame and prepare the visualization for frame-by-frame updates.

In [4]:
def render_umap_frame_for_timestamp(timestamp:pd.Timestamp, coord:np.ndarray, *, wshed_artifacts: dict, region=None) -> np.ndarray:
    return render_umap_panel(
        coord,
        wshed_artifacts["density"],
        extent=wshed_artifacts["extent"],
        wbounds=wshed_artifacts["wbounds"],
        barycenters=None,
        title="UMAP density map",
        timestamp=timestamp,
        region=region,
        figsize=UMAP_PANEL_FIGSIZE,
        dpi=UMAP_PANEL_DPI,
        title_fontsize=UMAP_PANEL_TITLE_FONTSIZE,
        axis_labelsize=UMAP_PANEL_AXIS_LABELSIZE,
        tick_labelsize=UMAP_PANEL_TICK_LABELSIZE,
    )


## Compose Side-by-Side or Top-Bottom Video Layouts

Implement layout functions to place the main video content and the UMAP panel either side by side or stacked vertically.

In [5]:
def blackout_unused_hive_half(dataset_frame: np.ndarray, ihl: str) -> np.ndarray:
    frame = np.asarray(dataset_frame).copy()
    if frame.ndim != 3 or frame.shape[0] < 2 or frame.shape[1] < 2:
        return frame

    original = frame.copy()
    height = frame.shape[0]
    width = frame.shape[1]
    midpoint = height // 2

    if ihl == "upper":
        frame[midpoint:, :, :] = 0
        ambient_x0 = int(width * (2700 / 3840))
        ambient_y0 = int(height * (2050 / 2160))
        frame[ambient_y0:height, ambient_x0:width, :] = original[ambient_y0:height, ambient_x0:width, :]
    elif ihl == "lower":
        frame[:midpoint, :, :] = 0
        timestamp_x0 = int(width * (1500 / 3840))
        timestamp_y0 = int(height * (980 / 2160))
        timestamp_x1 = int(width * (2350 / 3840))
        timestamp_y1 = int(height * (1165 / 2160))
        frame[timestamp_y0:timestamp_y1, timestamp_x0:timestamp_x1, :] = original[timestamp_y0:timestamp_y1, timestamp_x0:timestamp_x1, :]
    return frame


## Render Combined Video for a Single Dataset

Assemble the video frames with the UMAP panel, write the final video to disk, and expose this as a function that accepts a dataset identifier.

In [9]:
def render_dataset_video(dataset_name: str, *, layout: str = LAYOUT, fps: int = FPS) -> str:
    dataset_config = DATASET_CONFIGS[dataset_name]
    source_video_path = resolve_macos_alias(dataset_config["video_alias"])
    frame_interval = pd.Timedelta(minutes=20) # For all videos, frames are 20' apart
    umap_data = load_umap_data(DATASET_CONFIGS, PROJECTIONS_DIR, dataset_name)

    cap = cv2.VideoCapture(str(source_video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if FRAME_LIMIT is not None:
        frame_count = min(frame_count, FRAME_LIMIT)

    full_timeline = pd.date_range(start=dataset_config["start_ts"], periods=frame_count, freq=frame_interval)

    reader = imageio.get_reader(str(source_video_path))

    def frame_iterator(dts: pd.DatetimeIndex):
        progress = tqdm(total=frame_count, desc=f'"{dataset_name}" video frames', unit='frame')
        # compute video timestamps from known start and fixed cadence (20 minutes per frame)

        for idx, (_dt, video_frame) in enumerate(zip(dts, reader)):
            video_frame = np.asarray(video_frame)
            if video_frame.ndim == 2:
                video_frame = np.stack([video_frame] * 3, axis=-1)
            if video_frame.shape[2] == 4:
                video_frame = video_frame[:, :, :3]
            if SOURCE_VIDEO_SCALE != 1.0:
                new_width = max(1, int(video_frame.shape[1] * SOURCE_VIDEO_SCALE))
                new_height = max(1, int(video_frame.shape[0] * SOURCE_VIDEO_SCALE))
                video_frame = cv2.resize(video_frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
            video_frame = blackout_unused_hive_half(video_frame, dataset_config["ihl"] )

            # lookup a matching UMAP point for the _dt. If it does not exist, use None.
            umap_coord = None
            try:
                row = umap_data[umap_data["real_timestamps"] == _dt].iloc[0]
                umap_coord = np.array([row["umap_x"], row["umap_y"]], dtype=float)
            except IndexError:
                umap_coord = None

            # render the UMAP panel with the video timestamp and optional point overlay
            umap_frame = render_umap_frame_for_timestamp(
                _dt,
                umap_coord,
                wshed_artifacts=wshed_artifacts,
            )
            yield compose_panels(video_frame, umap_frame, layout=layout)
            progress.update(1)
        progress.close()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generate_video_name = f"{VIDEO_PREFIX}_{dataset_name}"
    try:
        output_path = generateVideoFromFrames(frame_iterator(full_timeline), dest=str(OUTPUT_DIR), name=generate_video_name, fps=fps)
    finally:
        reader.close()
    return output_path


In [10]:
dataset_name = 'hive1_upper'
umap_df = load_umap_data(dataset_cfg=DATASET_CONFIGS, projections_dir=PROJECTIONS_DIR, dataset_name=dataset_name)
print(f"Filtered UMAP timeline shape: {umap_df.shape}, start: {umap_df['real_timestamps'].min()}, end: {umap_df['real_timestamps'].max()}")

print(f"Different projection source_id counts:\n{umap_df.loc[:,'source_id'].value_counts()}")

# show a few entries
print('\nfiltered timeline head:')
print(umap_df.head())


Initial UMAP timeline shape: (10098, 4), start: 2024-09-14 00:40:00+00:00, end: 2024-10-28 04:40:00+00:00
Filtered UMAP timeline shape: (4516, 4), start: 2024-09-14 00:40:00+00:00, end: 2024-10-28 04:40:00+00:00
Different projection source_id counts:
1    4516
Name: source_id, dtype: int64

filtered timeline head:
     umap_x    umap_y           real_timestamps  source_id
0  1.552277  1.285454 2024-09-14 00:40:00+00:00          1
1  1.543904  1.265064 2024-09-14 00:50:00+00:00          1
2  1.521940  1.245905 2024-09-14 01:00:00+00:00          1
3  1.568719  1.318483 2024-09-14 01:10:00+00:00          1
4  1.542503  1.266071 2024-09-14 01:20:00+00:00          1


## Loop Over the Selected Dataset

Add batch-processing logic to generate videos for each dataset using the same rendering pipeline and configurable settings.

In [11]:
video_outputs = {}
for dataset_name in selected_datasets:
    video_outputs[dataset_name] = render_dataset_video(dataset_name, layout=LAYOUT, fps=FPS)

video_outputs


Initial UMAP timeline shape: (10098, 4), start: 2024-09-14 00:40:00+00:00, end: 2024-10-28 04:40:00+00:00


"hive1_upper" video frames:   0%|          | 0/3874 [00:00<?, ?frame/s]

{'hive1_upper': '/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/outputVideos/umap_video/dataset_umap_hive1_upper_2.mp4'}

## Verify Generated Videos

Confirm that the files were created successfully.

In [12]:
verified_outputs = {name: Path(path).exists() for name, path in video_outputs.items()}
verified_outputs

{'hive1_upper': True}